# ED Pathway Orchestrator — v12 with Synthetic Data Factory

In [ ]:
import os, time, math, json, random, zipfile
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Literal, Any, Tuple
import numpy as np, pandas as pd
SEED=4242
random.seed(SEED); np.random.seed(SEED)

def safe_clip(obj, lower=None, upper=None): return obj.clip(lower=lower, upper=upper)

def safe_assign(df: pd.DataFrame, col: str, values): df=df.copy(); df.loc[:,col]=values; return df

def assert_no_na(df: pd.DataFrame, cols):
    cols=[c for c in cols if c in df.columns]
    if not cols: return True
    na=df[cols].isna().sum()
    if int(na.sum())>0: raise ValueError('NA present: '+str({k:int(v) for k,v in na.items() if v>0}))
    return True
print('[env] ready')

In [ ]:
POLICY={'imaging':{'XR':{'embedded_ed':True,'requires_tech_transfer':True}, 'CT_resus':{'requires_ris_order':True}}, 'transfer':{'attending_required':True}, 'icu_joker':{'enabled':True,'trigger_occupancy':0.9,'extra_beds':1}}
CFG={'n_patients':500,'start_ts':pd.Timestamp('2025-08-10 08:00'),'arrival_hours':12,'prevalence':{'sepsis':0.18,'acs':0.12,'major_trauma':0.06,'stroke':0.05,'pregnant_syncope':0.03,'other':0.56},'sex_ratio_female':0.52,'pregnancy_age_range':(15,45),'lab_tat_minutes':{'cbc':(35,12),'bmp':(45,15),'lactate':(40,12),'crp':(55,15),'hs-ctnt':(50,15)},'xr_duration_min':(18,6),'ct_resus_duration_min':(25,8),'bed_capacity':{'ICU':2,'StepDown':4,'EDObs':6,'Ward':30},'nurse_slots':{'ICU':2,'StepDown':6,'EDObs':6,'Ward':30},'vents':2}
print('[cfg] set')

In [ ]:
def rand_time_within(start: pd.Timestamp, hours: int)->pd.Timestamp:
    return start + pd.Timedelta(minutes=float(np.random.uniform(0, hours*60)))

def bounded_norm(mean, sd, lo, hi):
    x=np.random.normal(mean, sd)
    return float(np.clip(x, lo, hi))

def triage_from_vitals(sbp, hr, spo2, gcs, stemi=False, trauma=False):
    si=hr/max(sbp,1)
    if stemi or trauma or sbp<90 or gcs<9 or spo2<88 or si>=1.0: return 'red'
    return 'yellow_green'

In [ ]:
def synth_vitals(cond:str):
    if cond=='sepsis':
        sbp=bounded_norm(95,18,60,150); hr=bounded_norm(110,20,60,170); spo2=bounded_norm(93,4,80,100); gcs=int(np.clip(np.random.normal(13.5,1.5),8,15))
    elif cond=='acs':
        sbp=bounded_norm(115,20,80,170); hr=bounded_norm(95,18,50,160); spo2=bounded_norm(95,3,85,100); gcs=int(np.clip(np.random.normal(14.5,0.8),10,15))
    elif cond=='major_trauma':
        sbp=bounded_norm(90,25,60,160); hr=bounded_norm(110,25,60,180); spo2=bounded_norm(92,6,75,100); gcs=int(np.clip(np.random.normal(12.0,3.0),3,15))
    elif cond=='stroke':
        sbp=bounded_norm(150,25,90,220); hr=bounded_norm(85,15,45,150); spo2=bounded_norm(95,3,85,100); gcs=int(np.clip(np.random.normal(13.0,2.0),6,15))
    else:
        sbp=bounded_norm(120,20,80,180); hr=bounded_norm(88,18,45,160); spo2=bounded_norm(96,2,88,100); gcs=int(np.clip(np.random.normal(14.5,1.0),10,15))
    return int(sbp), int(hr), int(spo2), int(gcs)

def synth_labs(cond:str, sex:str):
    labs={}
    if cond=='sepsis':
        labs['lactate']=float(np.round(np.random.lognormal(mean=math.log(2.2), sigma=0.3),2))
        labs['crp']=float(np.round(np.random.lognormal(mean=math.log(80), sigma=0.5),1))
    else:
        labs['lactate']=float(np.round(np.random.lognormal(mean=math.log(1.5), sigma=0.25),2))
        labs['crp']=float(np.round(np.random.lognormal(mean=math.log(12), sigma=0.6),1))
    if cond=='acs':
        if np.random.rand()<0.35:
            labs['hs-ctnt']=float(np.round(np.random.lognormal(mean=math.log(80), sigma=0.6),1))
        else:
            labs['hs-ctnt']=float(np.round(np.random.uniform(2.0,11.0),1))
    else:
        labs['hs-ctnt']=float(np.round(np.random.uniform(2.0,8.0),1))
    labs['cbc_hgb']=float(np.round(np.random.normal(13.5 if sex=='male' else 12.5, 1.2),1))
    labs['bmp_creat']=float(np.round(np.random.lognormal(mean=math.log(0.95), sigma=0.25),2))
    return labs

In [ ]:
def schedule_imaging(triage:str, stemi:bool, trauma:bool, arrival:pd.Timestamp):
    recs=[]
    if triage=='red' and (stemi or trauma):
        if trauma:
            dur=max(5,int(np.random.normal(*CFG['ct_resus_duration_min']))); start=arrival + pd.Timedelta(minutes=10)
            recs.append({'modality':'CT_resus','ordered_at':arrival,'start_at':start,'done_at':start+pd.Timedelta(minutes=dur)})
    else:
        dur=max(5,int(np.random.normal(*CFG['xr_duration_min']))); start=arrival + pd.Timedelta(minutes=int(np.random.uniform(20,45)))
        recs.append({'modality':'XR','ordered_at':arrival+pd.Timedelta(minutes=5),'start_at':start,'done_at':start+pd.Timedelta(minutes=dur)})
    return recs

In [ ]:
class BedState:
    def __init__(self, capacity:Dict[str,int], nurse_slots:Dict[str,int], vents:int):
        self.capacity=capacity.copy(); self.occ={k:0 for k in capacity}; self.nurse=nurse_slots.copy(); self.vents=vents; self.vent_use=0
    def admit(self, unit:str, needs_vent=False):
        if self.occ[unit]>=self.capacity[unit]: return False
        if self.nurse.get(unit,0)<=0: return False
        if needs_vent and self.vent_use>=self.vents: return False
        self.occ[unit]+=1; self.nurse[unit]=self.nurse.get(unit,0)-1
        if needs_vent: self.vent_use+=1
        return True
    def icu_joker(self):
        cap=self.capacity['ICU']; occ=self.occ['ICU']; ratio=1.0 if cap==0 else (occ/cap)
        if {'enabled':True}.get('enabled') and ratio>=0.9:
            self.capacity['ICU']+=1; self.nurse['ICU']=self.nurse.get('ICU',0)+1; return True
        return False

In [ ]:
def generate_dataset(cfg=CFG):
    ts_dir=f"/mnt/data/synth_run_{int(time.time())}"; os.makedirs(ts_dir, exist_ok=True)
    n=cfg['n_patients']; start=cfg['start_ts']
    conds=list(cfg['prevalence'].keys()); prev=np.array(list(cfg['prevalence'].values())); prev=prev/prev.sum()
    chosen=list(np.random.choice(conds,size=n,p=prev))
    sex=np.where(np.random.rand(n)<cfg['sex_ratio_female'],'female','male'); age=np.random.randint(18,95,size=n)
    pregnancy=np.zeros(n,dtype=bool)
    for i in range(n):
        if sex[i]=='female' and cfg['pregnancy_age_range'][0]<=age[i]<=cfg['pregnancy_age_range'][1]:
            if chosen[i] in ('pregnant_syncope',) and np.random.rand()<0.8: pregnancy[i]=True
            elif np.random.rand()<0.06: pregnancy[i]=True
    arrivals=[start + pd.Timedelta(minutes=float(np.random.uniform(0, cfg['arrival_hours']*60))) for _ in range(n)]
    beds=BedState(cfg['bed_capacity'], cfg['nurse_slots'], cfg['vents'])
    rows_pat=[]; rows_ems=[]; rows_labs=[]; rows_im=[]; rows_tx=[]
    for i in range(n):
        pid=f'P{i:05d}'; cond=chosen[i]; sbp,hr,spo2,gcs=synth_vitals(cond)
        stemi=(cond=='acs') and (np.random.rand()<0.15); trauma=(cond=='major_trauma')
        triage=triage_from_vitals(sbp,hr,spo2,gcs, stemi, trauma); arr=arrivals[i]
        rows_ems.append({'pid':pid,'arrival':arr,'sbp':sbp,'hr':hr,'spo2':spo2,'gcs':gcs,'stemi':stemi,'trauma':trauma,'triage':triage})
        lab_vals=synth_labs(cond, sex[i])
        for t,val in lab_vals.items():
            mean,sd = CFG['lab_tat_minutes'].get(t, (50,15)); tat=max(10,int(np.random.normal(mean,sd)))
            rows_labs.append({'pid':pid,'test':t,'ordered_at':arr+pd.Timedelta(minutes=5),'result_at':arr+pd.Timedelta(minutes=5+tat),'value':val})
        for rec in schedule_imaging(triage, stemi, trauma, arr):
            rec2=rec.copy(); rec2['pid']=pid; rows_im.append(rec2)
        needs_vent = True if (cond=='sepsis' and gcs<12) or (trauma and gcs<9) else False
        if stemi: target='ICU'
        elif triage=='red': target='StepDown'
        else: target='EDObs' if cond in ('sepsis','pregnant_syncope') else 'Ward'
        attending_ok=True
        success=beds.admit(target, needs_vent=needs_vent)
        joker_used=False; regional=False; boarded=False
        if not success:
            if target=='ICU' and beds.icu_joker():
                joker_used=True; success=beds.admit('ICU', needs_vent=needs_vent)
            if not success:
                if np.random.rand()<0.5: regional=True; success=True
                else: boarded=True
        rows_tx.append({'pid':pid,'requested_unit':target,'attending_ok':attending_ok,'needs_vent':needs_vent,'accepted':success,'regional':regional,'joker_used':joker_used,'boarded':boarded,'decision_at':arr+pd.Timedelta(minutes=int(np.random.uniform(15,90)))})
        rows_pat.append({'pid':pid,'sex':sex[i],'age':int(age[i]),'pregnancy':bool(pregnancy[i]),'condition':cond,'arrival':arr})
    patients=pd.DataFrame(rows_pat).sort_values('arrival'); ems=pd.DataFrame(rows_ems).sort_values('arrival')
    labs=pd.DataFrame(rows_labs).sort_values(['pid','result_at'])
    imaging=pd.DataFrame(rows_im).sort_values(['pid','start_at']) if rows_im else pd.DataFrame(columns=['pid','modality','ordered_at','start_at','done_at'])
    transfers=pd.DataFrame(rows_tx).sort_values(['decision_at'])
    for df, cols in [(patients,['pid','arrival']), (ems,['pid','arrival','triage']), (labs,['pid','test','ordered_at','result_at','value']), (transfers,['pid','requested_unit','decision_at'])]:
        assert_no_na(df, cols)
    summ={'n':int(len(patients)),'triage_red_pct':float((ems['triage']=='red').mean()),'board_rate':float(transfers['boarded'].mean()),'icu_accepts':int(((transfers['requested_unit']=='ICU') & (transfers['accepted']==True)).sum()),'regional_rate':float(transfers['regional'].mean())}
    for name,df in [('patients',patients),('ems',ems),('labs',labs),('imaging',imaging),('transfers',transfers)]:
        df.to_csv(os.path.join(ts_dir, f"{name}.csv"), index=False)
    with open(os.path.join(ts_dir,'summary.json'),'w') as f: json.dump(summ,f,indent=2, default=str)
    zip_path=os.path.join(ts_dir,'synth_artifacts.zip')
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
        for fn in ['patients.csv','ems.csv','labs.csv','imaging.csv','transfers.csv','summary.json']:
            zf.write(os.path.join(ts_dir,fn), arcname=fn)
    print('[synth] wrote to:', ts_dir); return ts_dir, summ

ts_dir, summ = generate_dataset(CFG)
print('Summary:', summ)

In [ ]:
print('Notebook saved at:', '/mnt/data/ED_Pathway_Orchestrator_AllInOne_v12_with_synth.ipynb')